# Load Data

In [39]:
from data_manager import DataManager

In [40]:
dm = DataManager('.data/')

In [41]:
dm._load_topic_model()

2025-11-04 16:24:42,083 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


In [42]:
dm._load_graphs()

In [43]:
dm._load_umap_embeddings()

# Query Dashboard

In [9]:
from dashboards import *

In [64]:
dashboard = VectorQueryDashboard(vector_model=dm)
dashboard.run(debug=False, host="0.0.0.0", port=8051)

# Author Dashboard

In [ ]:
class AuthorDashboard(BaseDashboard):
    def __init__(self, vector_model, title="Author Analysis Dashboard"):
        self.vector_model = vector_model
        super().__init__(title)
        self.layout = self.create_layout()

    def create_layout(self):
        """Define the layout for the dashboard."""
        return html.Div([
            # dcc.Store(id="query-store"),
            html.H2(self.title),
            
            html.Div([
                html.H3("Instructions"),
            
                html.P('''
                    T
                    
                '''),
                
                # html.P('The results will be visualized in two plots:'),
                # html.Ul([
                #     html.Li("A timeline plot showing the proportion of documents mentioning the query over time, broken down by author."),
                #     html.Li("A UMAP projection of the retrieved documents, allowing you to explore their distribution in embedding space. You can select points in the UMAP plot to see details of the corresponding documents below the plots."),
                #     html.Li('A list of documents selected through the UMAP plot.'),
                #     html.Li('A list of the top N retrieved documents, sorted by similarity to the query.')
                # ]),
                # html.P('You can select points in the UMAP plot to see details of the corresponding documents below the plots.'),
                
            ]),
            
            

            # html.Div([
            #     dcc.Input(
            #         id="query-text",
            #         type="text",
            #         placeholder="Enter query text...",
            #         style={"width": "50%", "marginRight": "10px"}
            #     ),
            #     dcc.Dropdown(
            #         id="num-results",
            #         options=[{"label": str(x), "value": x} for x in [10, 25, 50, 100, 500, 1000]],
            #         value=100,
            #         clearable=False,
            #         style={"width": "120px", "display": "inline-block", "marginRight": "10px"}
            #     ),
            #     html.Button("Run Query", id="run-query", n_clicks=0, style={"backgroundColor": "#0074D9", "color": "white"}),
            # ], style={"marginBottom": "20px", 'display' : 'flex', 'alignItems': 'center'}),

            # html.Div([
            #     dcc.Graph(id="timeline-plot", style={"flex": "2", "marginRight": "10px"}),
            #     dcc.Graph(id="umap-plot", style={"flex": "1"}),
            # ], style={
            #     "display": "flex",
            #     "flexDirection": "row",
            #     "gap": "20px",
            #     "alignItems": "stretch",
            #      "marginTop": "100px"
            # }),

            # html.Div(id="selected-results", style={"marginTop": "20px"}),

            # html.Div(id="results-list", style={"marginTop": "20px"})
        ], style={"margin": "40px"})

    def _register_callbacks(self):
        """Register Dash callbacks for interactivity."""

        @self.app.callback(
            [
                Output("timeline-plot", "figure"),
                Output("umap-plot", "figure"),
                Output("results-list", "children")
            ],
            [Input("run-query", "n_clicks")],
            [State("query-text", "value"), State("num-results", "value")]
        )
        def update_dashboard(n_clicks, query_text, n_results):
            """Run query and update all visualizations."""
            if not n_clicks or not query_text:
                # Empty default state
                empty_fig = px.scatter()
                return empty_fig, empty_fig, html.P("Enter a query and click 'Run Query'.")

            # Query the vector model
            q = self.vector_model.query(query_text, n_results)
            # df = pd.DataFrame(results)
            
            grouper = ['year', 'name']
            
            q_ids = q.chunkID.values.tolist()
            q_ilocs = q.index.values.tolist()
            
            
            
            all_articles = self.vector_model.flat_data.groupby(grouper, as_index=True).date.count()
            qg = q.groupby(grouper, as_index=True).distance.count()
            df = (qg/all_articles).reset_index().fillna(0).rename(columns={0:'mention_proportion'})
            

            # --- Timeline Plot ---
            timeline_fig = px.line(
                df,
                x="year",
                y='mention_proportion',
                hover_data=['name', 'mention_proportion'],
                color="name",
                title="Mentions Over Time"
            )
            timeline_fig.update_layout(height=400, width=1200, margin=dict(l=10, r=10, t=50, b=10))
            
            

            # --- UMAP Plot ---
            umap_embeds = self.vector_model.umap_embeddings[q_ilocs]
            umap_data = self.vector_model.flat_data.iloc[q_ilocs]
            umap_data = pd.concat(
                [
                    umap_data.reset_index(drop=True),
                    pd.DataFrame(umap_embeds, columns=['x', 'y'])
                ],
                axis=1
            )
            
            # display(umap_data.head())
            
            # print(umap_data.columns)
            
            umap_fig = px.scatter(
                umap_data,
                x="x",
                y="y",
                hover_data=["name", "title", 'year', 'month'],
                color="name",
                title="UMAP Projection",
                custom_data=['chunkID']
            )
            umap_fig.update_layout(height=400, margin=dict(l=10, r=10, t=50, b=10))
            
            results = q.sort_values(by="distance").to_dict(orient="records")

            # --- Results List ---
            results_children = [
                html.H3(f"Top {n_results} Results"),
                html.Ul([
                    html.Li([
                        html.B(f'{r["title"]} (by {r["name"]}; distance {r['distance']: .2f}): '),
                        html.Span(r["content"])
                    ]) for r in results
                ])
            ]

            return timeline_fig, umap_fig, results_children
        
        @self.app.callback(
            Output("selected-results", "children"),
            Input("umap-plot", "selectedData"),
            
        )
        def filter_texts(selectedData):
            if selectedData is None:
                return html.P("Select points in the UMAP plot to see details here.")
            
            # print(selectedData)
            
            indices = [p['customdata'][0] for p in selectedData['points']]
            # print(indices)
            
            df = self.vector_model.flat_data
            df = df[df['chunkID'].isin(indices)]
            
            
            
            return [
                html.H3(f"Selected {len(indices)} Results"),
                html.Ul([
                    html.Li([
                        html.B(f'{i["title"]} (by {i["name"]} on {i['date'].strftime('%d.%m.%Y')}): '),
                        html.Span(i["content"])
                    ]) for i in df.to_dict(orient="records")
                ])
            ]

# Topic Model Dashboard

In [65]:
t = TopicModelDashboard(data_manager=dm)
t.run(debug=False, host="0.0.0.0", port=8052)

[Div(children=[H3('Topic 0'), Pre(children='*) Oswald Spengler, Decline of the West, two vols. (New York: Alfred A. Knopf, 1928), vol. 2, p. 245. **) E. W. Hengstenberg, “The Jews and the Christian Church,” pp. 413-478 in Commentary on Ecclesiastes: With Other Treatises, trans. D. W. Simon (Edinburgh: T & T. Clark, 1860), p. 417.\n------\n*) Ralph Waldo Emerson, “Self Reliance,” pp. 25-53 in Essays, Orations and Lectures (London: William Tegg and Co., 1848), p. 29. **) Letter to Polly Stevenson, March 25, 1763, in Jared Sparks, ed., A Collection of the Familiar Letters and Miscellaneous Papers of Benjamin Franklin (Boston: C. Bowen, 1833), p. 78. ***) Thomas Carlyle, Later-Day Pamphlets (London: Chapman and Hall, 1958), p. 3. †) Thomas Carlyle, The French Revolution: A History, three vols. (Boston: Charles C. Little and James Brown, 1838), vol. 1, p. 35. ††) Thomas Carlyle, Heroes and Hero Worship (London: Chapman and Hall, 1840), p. 220. †††) Carlyle, French Revolution, vol. 1, p. 344

# Main App

In [ ]:
from dash import Dash, html, dcc
import dash

app = Dash(__name__, use_pages=True)
app.title = "My Multi-Dashboard App"

app.layout = html.Div([
    html.Nav([
        dcc.Link("Dashboard 1", href="/dashboard1"),
        " | ",
        dcc.Link("Dashboard 2", href="/dashboard2"),
    ], className="navbar"),

    html.Hr(),
    dash.page_container  # This renders the active page
])

if __name__ == "__main__":
    app.run_server(debug=True)
